In [ ]:
import pandas as pd
from pipeline import IndicatorSpec, build_feature_df
from data import fetch_binance_klines
from simulation.rules import Rule, RuleGroup, ALL, ANY
from simulation.rule_features import add_last_n_compare, add_cross_compare
from simulation.simulator import Strategy, SimConfig, run_simulation, align_timeframes
from plot_simulation import plot_simulation
from plotter import plot_interactive, PlotConfig, IndicatorSpec
from mtf_plot import make_plot_indicators, assert_plot_columns_exist
from features.ema_spreads import EmaSpreadSpec, add_ema_spread_features
from simulation.rules import Ctx
from analytics import (
    trades_to_frame,
    plot_balance_by_trade,
    plot_trade_pnl_bars,
    trade_summary_table
)
from plot_toggles import (
    PlotToggle,
    make_plot_indicators_from_toggles,
    assert_plot_columns_exist,
)


In [ ]:
symbol = "BTCUSDT"
market = "spot"
tz = "Asia/Karachi"

# Start earlier for EMA warmup.
fetch_start = "2026-01-01"
sim_start = "2026-05-06"
end = None

In [ ]:
RSI_DIV_CFG = {
    "length": 14,
    "pivot_lookback": 5,
    "min_rsi_delta": 2,

    # divergence filtering
    "os_level": 30,
    "ob_level": 70,
    "zone_mode": "cross",

    # panel styling
    "major_levels": [30, 70],
    "minor_levels": [20, 80],
    "show_zone_shading": True,

    # divergence labels
    "show_div_labels": True,
    "max_div_labels": 6,
    "label_font_size": 10,
    "label_xshift": 10,
    "label_yshift": 14,

    # main chart markers
    "mark_price": True,
    "price_marker_size": 18,
    "marker_offset_mult": 1.35,
}

In [ ]:
ind_1m = [
    IndicatorSpec("moving_average", tag="ema", config={
        "type": "ema",
        "periods": [50, 100, 150, 200, 250],
        "source": "close",
    }),

    IndicatorSpec("macd", tag="macd_8_21_5", config={
        "fast": 8,
        "slow": 21,
        "signal": 5,
    }),

    IndicatorSpec("rsi_divergence", tag="rsi14", config=RSI_DIV_CFG),
]


ind_5m = [
    IndicatorSpec("moving_average", tag="ema", config={
        "type": "ema",
        "periods": [50, 75, 100, 125, 150, 175, 200],
        "source": "close",
    }),

    IndicatorSpec("macd", tag="macd_8_21_5", config={
        "fast": 8,
        "slow": 21,
        "signal": 5,
    }),

    IndicatorSpec("rsi_divergence", tag="rsi14", config=RSI_DIV_CFG),
]


ind_15m = [
    IndicatorSpec("moving_average", tag="ema", config={
        "type": "ema",
        "periods": [50, 75, 100, 125, 150, 175, 200],
        "source": "close",
    }),

    IndicatorSpec("macd", tag="macd_8_21_5", config={
        "fast": 8,
        "slow": 21,
        "signal": 5,
    }),

    IndicatorSpec("rsi_divergence", tag="rsi14", config=RSI_DIV_CFG),
]

In [ ]:
# 1m base data
df1 = fetch_binance_klines(
    symbol=symbol,
    interval="1m",
    start=fetch_start,
    end=end,
    market=market,
    cache=True,
    cache_dir="data",
)

df1_feat, _, _ = build_feature_df(
    raw_df=df1,
    tz=tz,
    ma_windows=[],
    indicators=ind_1m,
)


# 5m filter data
df5 = fetch_binance_klines(
    symbol=symbol,
    interval="5m",
    start=fetch_start,
    end=end,
    market=market,
    cache=True,
    cache_dir="data",
)

df5_feat, _, _ = build_feature_df(
    raw_df=df5,
    tz=tz,
    ma_windows=[],
    indicators=ind_5m,
)


# 15m filter data
df15 = fetch_binance_klines(
    symbol=symbol,
    interval="15m",
    start=fetch_start,
    end=end,
    market=market,
    cache=True,
    cache_dir="data",
)

df15_feat, _, _ = build_feature_df(
    raw_df=df15,
    tz=tz,
    ma_windows=[],
    indicators=ind_15m,
)

merged = align_timeframes(
    base_df=df1_feat,
    other_dfs={
        "5m": df5_feat,
        "15m": df15_feat,
    },
    base_label="1m",
)

In [ ]:
for col in merged.columns:
    print(col)

In [ ]:
### 1 Minute Indicators ###
EMA50_1m = "1m__ema__EMA_50"
EMA100_1m = "1m__ema__EMA_100"
EMA150_1m = "1m__ema__EMA_150"
MACD_1M = "1m__macd_8_21_5__MACD"
MACD_SIGNAL_1M = "1m__macd_8_21_5__SIGNAL"
MACD_HIST_1M = "1m__macd_8_21_5__HIST"
BULL_DIV_1m = "1m__rsi14__BULL_DIV"
BEAR_DIV_1m = "1m__rsi14__BEAR_DIV"
BULL_START_RSI_1m = "1m__rsi14__BULL_START_RSI"
BEAR_START_RSI_1m = "1m__rsi14__BEAR_START_RSI"


### 5 Minute Indicators ###
EMA50_5m = "5m__ema__EMA_50"
EMA75_5m = "5m__ema__EMA_75"
EMA100_5m = "5m__ema__EMA_100"
EMA125_5m = "5m__ema__EMA_125"
EMA150_5m = "5m__ema__EMA_150"
EMA175_5m = "5m__ema__EMA_175"
EMA200_5m = "5m__ema__EMA_200"
MACD_5M = "5m__macd_8_21_5__MACD"
MACD_SIGNAL_5M = "5m__macd_8_21_5__SIGNAL"
MACD_HIST_5M = "5m__macd_8_21_5__HIST"
BULL_DIV_5m = "5m__rsi14__BULL_DIV"
BEAR_DIV_5m = "5m__rsi14__BEAR_DIV"
BULL_START_RSI_5m = "5m__rsi14__BULL_START_RSI"
BEAR_START_RSI_5m = "5m__rsi14__BEAR_START_RSI"


### 15 Minute Indicators ###
EMA50_15m = "15m__ema__EMA_50"
EMA75_15m = "15m__ema__EMA_75"
EMA100_15m = "15m__ema__EMA_100"
EMA125_15m = "15m__ema__EMA_125"
EMA150_15m = "15m__ema__EMA_150"
EMA175_15m = "15m__ema__EMA_175"
EMA200_15m = "15m__ema__EMA_200"
MACD_15M = "15m__macd_8_21_5__MACD"
MACD_SIGNAL_15M = "15m__macd_8_21_5__SIGNAL"
MACD_HIST_15M = "15m__macd_8_21_5__HIST"
BULL_DIV_15m = "15m__rsi14__BULL_DIV"
BEAR_DIV_15m = "15m__rsi14__BEAR_DIV"
BULL_START_RSI_15m = "15m__rsi14__BULL_START_RSI"
BEAR_START_RSI_15m = "15m__rsi14__BEAR_START_RSI"


LONG_EMA_FILTERS = [
    EMA50_1m,
    EMA100_1m,
    EMA150_1m,
    EMA100_5m,
    EMA100_15m,
]

In [ ]:
N_CONFIRM = 1             # number of consecutive confirmation candles required
MAX_WAIT_AFTER_CROSS = 30   # how many candles after the cross we are willing to wait
MACD_THRESHOLD_1M = 10
MACD_THRESHOLD_5M = 10
MACD_THRESHOLD_15M = 50 

MIN_MACD_DIFF_1M = 5
MIN_MACD_DIFF_5M = 5
MIN_MACD_DIFF_15M = 5

DIV_LOOKBACK = 5

###################### long rules #########################
open_rules_long = ALL(
    Rule(
        "Current close above all EMA filters",
        lambda c: c.close_above_all(LONG_EMA_FILTERS)
    ),

    Rule(
        f"Last {N_CONFIRM} closes above all EMA filters",
        lambda c: c.last_n_closes_above_all(LONG_EMA_FILTERS, n=N_CONFIRM)
    ),

    Rule(
        f"Last {N_CONFIRM} candles green",
        lambda c: c.consecutive_green(n=N_CONFIRM)
    ),

    # Optional but recommended: 1m EMA ribbon should be bullish
    Rule(
        "1m EMA ribbon bullish: EMA50 > EMA100 > EMA150",
        lambda c: (
            c.gt(EMA50_1m, EMA100_1m)
            and c.gt(EMA100_1m, EMA150_1m)
        )
    ),

        # Optional but recommended: 1m EMA ribbon should be bullish
    Rule(
        "1m EMA ribbon bullish: EMA50 > EMA100 > EMA150",
        lambda c: (
            c.gt(EMA50_5m, EMA100_5m)
            and c.gt(EMA100_5m, EMA150_5m)
        )
    ),

        # Optional but recommended: 1m EMA ribbon should be bullish
    Rule(
        "1m EMA ribbon bullish: EMA50 > EMA100 > EMA150",
        lambda c: (
            c.gt(EMA50_15m, EMA100_15m)
            and c.gt(EMA100_15m, EMA150_15m)
        )
    ),

    Rule(
        "1m MACD above threshold",
        lambda c: c.gt(MACD_1M, MACD_THRESHOLD_1M)
    ),
    
    Rule(
        "5m MACD above threshold",
        lambda c: c.gt(MACD_5M, MACD_THRESHOLD_5M)
    ),
    
    Rule(
        "15m MACD above threshold",
        lambda c: c.gt(MACD_15M, MACD_THRESHOLD_15M)
    ),

    Rule(
        "1m MACD - Signal above minimum",
        lambda c: c.gt(MACD_HIST_1M, MIN_MACD_DIFF_1M)
    ),
    
    Rule(
        "5m MACD - Signal above minimum",
        lambda c: c.gt(MACD_HIST_5M, MIN_MACD_DIFF_5M)
    ),

    Rule(
        "15m MACD - Signal above minimum",
        lambda c: c.gt(MACD_HIST_15M, MIN_MACD_DIFF_15M)
    ),

    # Rule(
    #     f"No bearish RSI divergence in last {DIV_LOOKBACK} candles",
    #     lambda c: not c.flag_recent(BEAR_DIV_1m, lookback=1200)
    # ),
    #     Rule(
    #     f"No bearish RSI divergence in last {DIV_LOOKBACK} candles",
    #     lambda c: not c.flag_recent(BEAR_DIV_15m, lookback=10)
    # ),
    #     Rule(
    #     f"No bearish RSI divergence in last {DIV_LOOKBACK} candles",
    #     lambda c: not c.flag_recent(BEAR_DIV_15m, lookback=10)
    # ),
)




# open_rules_long = ALL(
#     # Rule(
#     #     f"Close crossed above EMA100 and later {N_CONFIRM} green confirmations",
#     #     lambda c: c.cross_up_with_later_confirm(
#     #         left="close",
#     #         right=EMA100_15m,

#     #         # Look back for a previous cross.
#     #         max_bars_since_cross=MAX_WAIT_AFTER_CROSS,

#     #         # Require last N_CONFIRM candles to confirm.
#     #         confirm_bars=N_CONFIRM,
#     #         confirm_pattern="green",

#     #         # Each confirmation candle must still close above EMA50.
#     #         require_same_side=True,

#     #         # Price must not have crossed back below EMA50 after the original cross.
#     #         require_no_opposite_cross=True,
#     #     )
#     # ),

    

    
# )

#EMA75_15m = "15m__ema__EMA_75"

close_rules_long = ALL(
    Rule(
        "Close below EMA100",
        lambda c: c.lt("close", EMA100_15m)
    ),
)

###################### short rules #########################

# open_rules_short = ALL(
#     Rule(
#         f"Close crossed below EMA50 and later {N_CONFIRM} red confirmations",
#         lambda c: c.cross_down_with_later_confirm(
#             left="close",
#             right=EMA50,
#             max_bars_since_cross=MAX_WAIT_AFTER_CROSS,
#             confirm_bars=N_CONFIRM,
#             confirm_pattern="red",
#             require_same_side=True,
#             require_no_opposite_cross=True,
#         )
#     ),
# )


# close_rules_short = ALL(
#     Rule(
#         "Close above EMA50",
#         lambda c: c.gt("close", EMA50)
#     ),
# )

In [ ]:
strategy = Strategy(
    open_rules_long=open_rules_long,
    close_rules_long=close_rules_long,

    # allow_short=True,
    # open_rules_short=open_rules_short,
    # close_rules_short=close_rules_short,
)

res = run_simulation(
    df=merged,
    strategy=strategy,
    cfg=SimConfig(
        initial_cash=10_000,
        max_open_trades=1,
        fee_bps=10,
        slippage_bps=1,
        # sim_start=sim_start,
        sim_start="2026-01-01",
        sim_end="2026-05-09",
        sim_tz=tz,
    ),
    time_col="t",
    price_col="close",
)

res.stats #, res.events.head(), res.equity_curve.tail()

In [ ]:
# fig1 = plot_balance_by_trade(res.trades, initial_cash=10_000, interval="5m")
# fig1.show()

# fig2 = plot_trade_pnl_bars(res.trades, initial_cash=10_000, interval="1m")
# fig2.show()


In [ ]:
trade_summary_table(res.trades, initial_cash=10_000, interval="1m")

In [ ]:
# plot_indicators = make_plot_indicators(
#     base_specs=ind_1m,
#     mtf_specs={
#         "5m": ind_5m,
#         "15m": ind_15m,
#     },
#     mtf_include={
#         "5m": ["macd_8_21_5"],
#         "15m": ["macd_8_21_5"],
#     }
# )

In [ ]:
indicators_by_tf = {
    "1m": ind_1m,
    "5m": ind_5m,
    "15m": ind_15m,
}

plot_toggles = [
    PlotToggle("1m", "ema", visible=True),
    PlotToggle("5m", "ema", visible=True),
    PlotToggle("15m", "ema", visible=True),


    # Available in legend, hidden by default
    PlotToggle("1m", "macd_8_21_5", visible="legendonly"),
    PlotToggle("5m", "macd_8_21_5", visible="legendonly"),
    PlotToggle("15m", "macd_8_21_5", visible="legendonly"),

    # Not plotted at all, but still can be used in rules
    PlotToggle("1m", "rsi14", visible=False),
    PlotToggle("5m", "rsi14", visible=False),
    PlotToggle("15m", "rsi14", visible=False),
]

plot_indicators = make_plot_indicators_from_toggles(
    indicators_by_tf=indicators_by_tf,
    toggles=plot_toggles,
)

assert_plot_columns_exist(merged, plot_indicators)

In [ ]:
plot_simulation(
    df_raw=merged,
    symbol=symbol,
    interval="1m",
    market=market,
    trades=res.trades,
    ma_windows=[],
    indicators=plot_indicators,
    plot_cfg=PlotConfig(
        tz=tz,
        # date_from="2026-05-06",
        # date_to="2026-05-10",
        date_from="2026-05-10",
        date_to="2026-05-15",
        height=1400,
        width=1900,
    ),
)

# plot_simulation(
#     df_raw=df1_feat,  # (ideally raw candles df; but if yours works as-is, ok)
#     symbol=symbol,
#     interval="5m",
#     market=market,
#     trades=res.trades,
#     ma_windows=[20],
#     indicators=ind_5m,
#     plot_cfg=PlotConfig(
#         tz="Asia/Karachi",
#         date_from="2026-02-01",
#         date_to="2026-02-05",
#         height=1400,
#         width=1900
#     ),
# )